In [1]:

import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "prod_ml").is_dir()
)

sys.path.insert(0, str(root / "src"))



In [2]:
from prod_ml.core.interfaces.data_provider import TrainingDataProvider
from prod_ml.services import ValidatorService, FeatureService, DefaultTrainingService

import pandas as pd

In [3]:

class DataSource(TrainingDataProvider):

    def get_training_dataset(self) -> pd.DataFrame:
        return pd.read_csv("../../data/heart.csv")


In [4]:
datasource=DataSource()
df =datasource.get_training_dataset()


In [5]:
from prod_ml.domains.heart_disease.services.schema_validator import SchemaValidator

validator_service=ValidatorService(
    validators=[
                SchemaValidator()
            ]
)
result =validator_service.validate(df)
print(result)

ValidationResult(passed=True, errors=[])


In [6]:
df[df['RestingBP']<=0].head()


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
449,55,M,NAP,0,0,0,Normal,155,N,1.5,Flat,1


In [7]:
from prod_ml.domains.heart_disease.services import (
    HeartCompleteFeatureTransformation,
    CategoricalFeatureEncoder
)


feature_service=FeatureService(
    transformers=[
        HeartCompleteFeatureTransformation(),
        CategoricalFeatureEncoder()
    ]
)

df=feature_service.fit_transform(df)
df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Age_MaxHR_Ratio,Stress_Index,Cholesterol_missing,...,RestingECG_ST,ExerciseAngina_0,ExerciseAngina_1,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up,Age_Group_Adult,Age_Group_Middle,Age_Group_Senior,Age_Group_Young
0,40,140.0,289.0,0,172,0.0,0,4.300000,0.0,0,...,0,1,0,0,0,1,0,0,0,1
1,49,160.0,180.0,0,156,1.0,1,3.183673,0.0,0,...,0,1,0,0,1,0,0,1,0,0
2,37,130.0,283.0,0,98,0.0,0,2.648649,0.0,0,...,1,1,0,0,0,1,0,0,0,1
3,48,138.0,214.0,0,108,1.5,1,2.250000,1.5,0,...,0,0,1,0,1,0,0,1,0,0
4,54,150.0,195.0,0,122,0.0,0,2.259259,0.0,0,...,0,1,0,0,0,1,1,0,0,0


In [8]:
from prod_ml.domains.heart_disease.services.data_quality_validator import DataQualityValidator

validator_service=ValidatorService(
    validators=[

                DataQualityValidator()
            ]
)
result =validator_service.validate(df)
print(result)

ValidationResult(passed=True, errors=[])


In [12]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


models = {

'Logistic Regression':
Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
]),


'SVM':
Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(
        probability=True,
        C=1,
        kernel='rbf'
    ))
]),


'Random Forest':
RandomForestClassifier(
    n_estimators=300,
    random_state=42
),


'XGBoost':
XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    random_state=42
)

}


In [13]:

from sklearn.metrics import *

results = []

for name, model in models.items():

    model.fit(X_train,y_train)

    pred = model.predict(X_test)

    prob = model.predict_proba(X_test)[:,1]


    results.append({
        'Model':name,
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred),
        'Recall':recall_score(y_test,pred),
        'F1':f1_score(y_test,pred),
        'ROC_AUC':roc_auc_score(y_test,prob)
    })


results_df = pd.DataFrame(results)

results_df.sort_values(
    'ROC_AUC',
    ascending=False
)

/home/feroz/Desktop/python/prod_ml_app/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
1,SVM,0.885870,0.871560,0.931373,0.900474,0.945361
2,Random Forest,0.902174,0.896226,0.931373,0.913462,0.935736
0,Logistic Regression,0.907609,0.912621,0.921569,0.917073,0.933764
3,XGBoost,0.880435,0.908163,0.872549,0.890000,0.926949
